# Automated Data Quality Checks

## Purpose

This notebook performs automated data quality validations across the Auto Insurance Claims & Risk Data Platform.

The objective is to detect data integrity issues, validate business rules, and ensure that downstream analytical tables contain consistent and reliable information.

## Key Objectives

- **Completeness:** Detect missing values in critical fields.
- **Uniqueness:** Verify that policy identifiers are unique.
- **Validity:** Check exposure, claim counts, and financial claim amounts.
- **Referential Integrity:** Ensure that valid claims reference existing policies.
- **Reconciliation:** Verify consistency between data processing layers.
- **Pipeline Reliability:** Fail execution when critical data quality rules are violated.

## Expected Outcome

The notebook produces automated validation results and raises an exception if a critical data quality check fails.

**Note:** Known discrepancies between reported claim counts and available financial claim records are monitored separately and are not treated as unexpected pipeline failures.

In [0]:
# Import PySpark SQL functions
from pyspark.sql import functions as F

# Load validated Silver tables
df_policies = spark.table("auto_insurance.silver.policies")
df_claims = spark.table("auto_insurance.silver.claims")
df_claims_orphan = spark.table("auto_insurance.silver.claims_orphan")

# Display source record counts
print("Silver policies:", df_policies.count())
print("Silver valid claims:", df_claims.count())
print("Silver orphan claims:", df_claims_orphan.count())

Silver policies: 678013
Silver valid claims: 26444
Silver orphan claims: 195


In [0]:
# Store data quality check results
dq_results = []

# Register a data quality check result
def record_dq_check(check_name, actual_value, expected_value):
    passed = actual_value == expected_value

    dq_results.append({
        "check_name": check_name,
        "actual_value": actual_value,
        "expected_value": expected_value,
        "status": "PASS" if passed else "FAIL"
    })

    print(
        f"[{'PASS' if passed else 'FAIL'}] "
        f"{check_name}: actual={actual_value}, expected={expected_value}"
    )

## Silver Policies — Data Quality Checks

### Purpose

Validate the integrity and reliability of policy records in `auto_insurance.silver.policies` before they are used for downstream analytics.

### Validation Rules

- **Completeness:** Ensure that every policy has a non-null `policy_id`.
- **Uniqueness:** Verify that each `policy_id` appears only once.
- **Exposure Validity:** Ensure that `exposure` is not null and is greater than zero.
- **Claim Count Validity:** Ensure that `claim_count` is not null and is non-negative.

### Expected Outcome

All four data quality checks should return zero invalid records and a **PASS** status.

Any failed check indicates a potential data integrity issue that must be investigated before downstream processing.

**Note:** Validation results are collected for reporting and will be evaluated together at the end of the notebook to determine whether the pipeline should fail.

In [0]:
# Check for missing policy identifiers
null_policy_ids = df_policies.filter(
    F.col("policy_id").isNull()
).count()

record_dq_check(
    "policies_null_policy_ids",
    null_policy_ids,
    0
)

# Check for duplicate policy identifiers
duplicate_policy_ids = (
    df_policies
    .groupBy("policy_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

record_dq_check(
    "policies_duplicate_policy_ids",
    duplicate_policy_ids,
    0
)

# Check for invalid insurance exposure
invalid_exposure = df_policies.filter(
    F.col("exposure").isNull()
    | (F.col("exposure") <= 0)
).count()

record_dq_check(
    "policies_invalid_exposure",
    invalid_exposure,
    0
)

# Check for invalid reported claim counts
invalid_claim_counts = df_policies.filter(
    F.col("claim_count").isNull()
    | (F.col("claim_count") < 0)
).count()

record_dq_check(
    "policies_invalid_claim_counts",
    invalid_claim_counts,
    0
)

[PASS] policies_null_policy_ids: actual=0, expected=0
[PASS] policies_duplicate_policy_ids: actual=0, expected=0
[PASS] policies_invalid_exposure: actual=0, expected=0
[PASS] policies_invalid_claim_counts: actual=0, expected=0


In [0]:
# Convert collected validation results into a Spark DataFrame
df_dq_results = spark.createDataFrame(dq_results)

# Display the validation summary
display(df_dq_results)

actual_value,check_name,expected_value,status
0,policies_null_policy_ids,0,PASS
0,policies_duplicate_policy_ids,0,PASS
0,policies_invalid_exposure,0,PASS
0,policies_invalid_claim_counts,0,PASS


## Silver Claims — Data Quality Checks

### Purpose

Validate the completeness, financial validity, and referential integrity of claim records in `auto_insurance.silver.claims`.

The objective is to ensure that all financial claims are valid and correctly linked to existing insurance policies before they are used in Gold-layer analytics.

### Validation Rules

- **Completeness:** Ensure that every claim has a non-null `policy_id`.
- **Financial Validity:** Verify that `claim_amount` is not null, is not NaN, and is greater than zero.
- **Referential Integrity:** Ensure that every claim references an existing policy in `auto_insurance.silver.policies`.

### Expected Outcome

All three data quality checks should return zero invalid records and a **PASS** status.

Any failed check indicates a potential data quality or referential integrity issue that requires investigation.

**Note:** Multiple claims may reference the same `policy_id`, as one insurance policy can have multiple claim records. Therefore, duplicate policy identifiers in the Claims table are not considered a data quality issue.

In [0]:
# Check for missing claim policy identifiers
null_claim_policy_ids = df_claims.filter(
    F.col("policy_id").isNull()
).count()

record_dq_check(
    "claims_null_policy_ids",
    null_claim_policy_ids,
    0
)

# Check for missing or invalid financial claim amounts
invalid_claim_amounts = df_claims.filter(
    F.col("claim_amount").isNull()
    | F.isnan("claim_amount")
    | (F.col("claim_amount") <= 0)
).count()

record_dq_check(
    "claims_invalid_claim_amounts",
    invalid_claim_amounts,
    0
)

# Check for claims referencing nonexistent policies
claims_without_policy = (
    df_claims
    .select("policy_id")
    .join(
        df_policies.select("policy_id"),
        on="policy_id",
        how="left_anti"
    )
    .count()
)

record_dq_check(
    "claims_without_valid_policy",
    claims_without_policy,
    0
)

[PASS] claims_null_policy_ids: actual=0, expected=0
[PASS] claims_invalid_claim_amounts: actual=0, expected=0
[PASS] claims_without_valid_policy: actual=0, expected=0


In [0]:
# Check whether quarantined claims unexpectedly match a valid policy
orphan_claims_with_policy = (
    df_claims_orphan
    .select("policy_id")
    .join(
        df_policies.select("policy_id"),
        on="policy_id",
        how="left_semi"
    )
    .count()
)

record_dq_check(
    "orphan_claims_with_valid_policy",
    orphan_claims_with_policy,
    0
)

# Check whether all orphan claims have the expected quarantine reason
invalid_quarantine_reasons = df_claims_orphan.filter(
    F.col("quarantine_reason").isNull()
    | (F.col("quarantine_reason") != "POLICY_NOT_FOUND")
).count()

record_dq_check(
    "orphan_claims_invalid_quarantine_reason",
    invalid_quarantine_reasons,
    0
)

[PASS] orphan_claims_with_valid_policy: actual=0, expected=0
[PASS] orphan_claims_invalid_quarantine_reason: actual=0, expected=0


In [0]:
# Display all collected Silver data quality check results
df_dq_results = spark.createDataFrame(dq_results)

display(
    df_dq_results.orderBy("check_name")
)

# Summarize validation statuses
display(
    df_dq_results.groupBy("status").count()
)

actual_value,check_name,expected_value,status
0,claims_invalid_claim_amounts,0,PASS
0,claims_null_policy_ids,0,PASS
0,claims_without_valid_policy,0,PASS
0,orphan_claims_invalid_quarantine_reason,0,PASS
0,orphan_claims_with_valid_policy,0,PASS
0,policies_duplicate_policy_ids,0,PASS
0,policies_invalid_claim_counts,0,PASS
0,policies_invalid_exposure,0,PASS
0,policies_null_policy_ids,0,PASS


status,count
PASS,9


## Bronze-to-Silver Reconciliation

### Purpose

Verify that all records ingested into the Bronze layer are accounted for after Silver-layer transformations.

### Validation Rules

- **Policy Reconciliation:** The number of Bronze policy records must match the number of Silver policy records.
- **Claims Reconciliation:** The number of Bronze claim records must equal the combined number of valid Silver claims and quarantined orphan claims.

### Expected Outcome

Both reconciliation checks should return a **PASS** status.

These validations ensure record-level completeness across processing layers, including the preservation of unmatched claims in the quarantine table.

**Note:** Record-count reconciliation verifies completeness at the aggregate level but does not guarantee that individual records are unchanged.

In [0]:
# Load raw Bronze datasets
df_bronze_policies = spark.table(
    "auto_insurance.bronze.policies_raw"
)

df_bronze_claims = spark.table(
    "auto_insurance.bronze.claims_raw"
)

In [0]:
# Reconcile Bronze and Silver policy counts
bronze_policy_count = df_bronze_policies.count()
silver_policy_count = df_policies.count()

record_dq_check(
    "bronze_silver_policy_count_reconciliation",
    silver_policy_count,
    bronze_policy_count
)

# Reconcile Bronze claims with valid and quarantined Silver claims
bronze_claim_count = df_bronze_claims.count()

silver_total_claim_count = (
    df_claims.count() + df_claims_orphan.count()
)

record_dq_check(
    "bronze_silver_claim_count_reconciliation",
    silver_total_claim_count,
    bronze_claim_count
)

[PASS] bronze_silver_policy_count_reconciliation: actual=678013, expected=678013
[PASS] bronze_silver_claim_count_reconciliation: actual=26639, expected=26639


In [0]:
# Display the current data quality summary
df_dq_results = spark.createDataFrame(dq_results)

display(
    df_dq_results.groupBy("status").count()
)

status,count
PASS,11


## Gold Layer — Data Quality Checks

### Purpose

Validate the completeness and consistency of Gold-layer analytical tables against their Silver-layer source data.

### Validation Rules

- **Table Availability:** Ensure that all seven expected Gold tables exist and can be queried.
- **Policy Reconciliation:** Verify that frequency-based Gold tables preserve the total policy count.
- **Reported Claims Reconciliation:** Verify that frequency-based Gold tables preserve the total number of reported claims.
- **Exposure Reconciliation:** Verify that aggregated insurance exposure matches the Silver source.
- **Financial Claims Reconciliation:** Verify that severity and cost tables preserve recorded financial claim counts and amounts.

### Expected Outcome

All Gold-layer tables should reconcile with their Silver sources.

Any unexpected discrepancy should be reported as a failed data quality check.

**Note:** Financial claim records are incomplete relative to reported claim counts. Gold financial metrics are therefore validated against available Silver claim records, not against the total number of reported claims.

In [0]:
# Calculate reference metrics from Silver policies
silver_policy_metrics = df_policies.agg(
    F.count("*").alias("total_policies"),
    F.sum("claim_count").alias("total_claims"),
    F.sum("exposure").alias("total_exposure")
).first()

# Calculate reference metrics from Silver financial claims
silver_claim_metrics = df_claims.agg(
    F.count("*").alias("recorded_claims"),
    F.sum("claim_amount").alias("total_claim_amount")
).first()

print("Silver reference metrics calculated successfully.")

Silver reference metrics calculated successfully.


In [0]:
# Define frequency-based Gold tables
frequency_tables = [
    "risk_by_driver_age",
    "risk_by_vehicle_age",
    "risk_by_region",
    "risk_by_vehicle_power",
    "risk_by_vehicle_fuel"
]

# Validate aggregated frequency metrics
for table_name in frequency_tables:
    df_gold = spark.table(
        f"auto_insurance.gold.{table_name}"
    )

    gold_metrics = df_gold.agg(
        F.sum("total_policies").alias("total_policies"),
        F.sum("total_claims").alias("total_claims"),
        F.sum("total_exposure").alias("total_exposure")
    ).first()

    record_dq_check(
        f"{table_name}_policy_count",
        gold_metrics["total_policies"],
        silver_policy_metrics["total_policies"]
    )

    record_dq_check(
        f"{table_name}_reported_claims",
        gold_metrics["total_claims"],
        silver_policy_metrics["total_claims"]
    )

    # Compare exposure totals with floating-point tolerance
    exposure_difference = abs(
        gold_metrics["total_exposure"]
        - silver_policy_metrics["total_exposure"]
    )

    record_dq_check(
        f"{table_name}_exposure_reconciliation",
        exposure_difference < 0.001,
        True
    )

[PASS] risk_by_driver_age_policy_count: actual=678013, expected=678013
[PASS] risk_by_driver_age_reported_claims: actual=36102, expected=36102
[PASS] risk_by_driver_age_exposure_reconciliation: actual=True, expected=True
[PASS] risk_by_vehicle_age_policy_count: actual=678013, expected=678013
[PASS] risk_by_vehicle_age_reported_claims: actual=36102, expected=36102
[PASS] risk_by_vehicle_age_exposure_reconciliation: actual=True, expected=True
[PASS] risk_by_region_policy_count: actual=678013, expected=678013
[PASS] risk_by_region_reported_claims: actual=36102, expected=36102
[PASS] risk_by_region_exposure_reconciliation: actual=True, expected=True
[PASS] risk_by_vehicle_power_policy_count: actual=678013, expected=678013
[PASS] risk_by_vehicle_power_reported_claims: actual=36102, expected=36102
[PASS] risk_by_vehicle_power_exposure_reconciliation: actual=True, expected=True
[PASS] risk_by_vehicle_fuel_policy_count: actual=678013, expected=678013
[PASS] risk_by_vehicle_fuel_reported_claims

In [0]:
# Validate financial claim severity table
df_severity_gold = spark.table(
    "auto_insurance.gold.severity_by_driver_age"
)

severity_metrics = df_severity_gold.agg(
    F.sum("recorded_claims").alias("recorded_claims"),
    F.sum("total_claim_amount").alias("total_claim_amount")
).first()

record_dq_check(
    "severity_recorded_claims",
    severity_metrics["recorded_claims"],
    silver_claim_metrics["recorded_claims"]
)

# Allow a tolerance because severity amounts were rounded before persistence
record_dq_check(
    "severity_claim_amount",
    abs(
        severity_metrics["total_claim_amount"]
        - silver_claim_metrics["total_claim_amount"]
    ) < 0.1,
    True
)

# Validate observed claims cost table
df_cost_gold = spark.table(
    "auto_insurance.gold.claims_cost_by_driver_age"
)

cost_metrics = df_cost_gold.agg(
    F.sum("total_policies").alias("total_policies"),
    F.sum("total_claims").alias("total_claims"),
    F.sum("recorded_claims").alias("recorded_claims"),
    F.sum("total_exposure").alias("total_exposure"),
    F.sum("total_claim_amount").alias("total_claim_amount")
).first()

record_dq_check(
    "cost_policy_count",
    cost_metrics["total_policies"],
    silver_policy_metrics["total_policies"]
)

record_dq_check(
    "cost_reported_claims",
    cost_metrics["total_claims"],
    silver_policy_metrics["total_claims"]
)

record_dq_check(
    "cost_recorded_claims",
    cost_metrics["recorded_claims"],
    silver_claim_metrics["recorded_claims"]
)

record_dq_check(
    "cost_exposure",
    abs(
        cost_metrics["total_exposure"]
        - silver_policy_metrics["total_exposure"]
    ) < 0.001,
    True
)

record_dq_check(
    "cost_claim_amount",
    abs(
        cost_metrics["total_claim_amount"]
        - silver_claim_metrics["total_claim_amount"]
    ) < 0.01,
    True
)

[PASS] severity_recorded_claims: actual=26444, expected=26444
[PASS] severity_claim_amount: actual=True, expected=True
[PASS] cost_policy_count: actual=678013, expected=678013
[PASS] cost_reported_claims: actual=36102, expected=36102
[PASS] cost_recorded_claims: actual=26444, expected=26444
[PASS] cost_exposure: actual=True, expected=True
[PASS] cost_claim_amount: actual=True, expected=True


In [0]:
# Normalize data quality results for Spark schema inference
normalized_dq_results = [
    {
        "check_name": result["check_name"],
        "actual_value": str(result["actual_value"]),
        "expected_value": str(result["expected_value"]),
        "status": result["status"]
    }
    for result in dq_results
]

# Create the DataFrame with consistent data types
df_dq_results = spark.createDataFrame(normalized_dq_results)

# Display the updated validation summary
display(
    df_dq_results.groupBy("status").count()
)

status,count
PASS,33


## Claims Reconciliation Monitoring

### Purpose

Monitor known differences between reported policy claim counts and available financial claim records.

The source dataset contains incomplete claim-level financial coverage. These differences are tracked as data quality observations rather than pipeline failures.

### Monitoring Metrics

- **Reported Claims:** Total claim count reported in Silver policies.
- **Recorded Financial Claims:** Number of valid claim records available in Silver.
- **Missing Financial Claim Records:** Difference between reported claims and valid financial claim records.
- **Orphan Claims:** Financial claim records that cannot be linked to an existing policy.

### Expected Behavior

Monitoring metrics are displayed for transparency and investigation.

Known source-data discrepancies do not automatically fail the pipeline. Critical structural and reconciliation validations remain responsible for determining pipeline success or failure.

In [0]:
# Monitor reported claims and available financial claim records
reported_claims = int(
    silver_policy_metrics["total_claims"]
)

recorded_financial_claims = int(
    silver_claim_metrics["recorded_claims"]
)

orphan_claims = df_claims_orphan.count()

# Calculate the difference in available financial coverage
financial_claim_gap = (
    reported_claims - recorded_financial_claims
)

# Build monitoring metrics
claims_monitoring = [
    ("reported_claims", reported_claims),
    ("recorded_financial_claims", recorded_financial_claims),
    ("financial_claim_gap", financial_claim_gap),
    ("orphan_claims", orphan_claims)
]

df_claims_monitoring = spark.createDataFrame(
    claims_monitoring,
    ["metric", "value"]
)

display(df_claims_monitoring)

metric,value
reported_claims,36102
recorded_financial_claims,26444
financial_claim_gap,9658
orphan_claims,195


## Final Data Quality Report and Pipeline Validation

### Purpose

Consolidate all data quality validation results and determine whether the pipeline meets the required quality standards.

### Validation Logic

- Display the complete data quality report.
- Calculate the total number of passed and failed checks.
- Mark the pipeline as successful when all critical validations pass.
- Raise an exception when one or more data quality checks fail.

### Pipeline Integration

This notebook is designed to run as the final validation task in a Databricks Job, after Bronze ingestion, Silver transformations, reconciliation, and Gold analytics.

A failed validation raises an exception, allowing Databricks Jobs to identify the task as failed.

**Note:** Known financial claim coverage discrepancies are monitored separately and do not automatically trigger pipeline failure.

In [0]:
# Build the final data quality report
normalized_dq_results = [
    {
        "check_name": result["check_name"],
        "actual_value": str(result["actual_value"]),
        "expected_value": str(result["expected_value"]),
        "status": result["status"]
    }
    for result in dq_results
]

df_dq_results = spark.createDataFrame(normalized_dq_results)

# Display detailed results
display(df_dq_results.orderBy("check_name"))

# Calculate validation summary
total_checks = len(dq_results)

failed_checks = [
    result for result in dq_results
    if result["status"] == "FAIL"
]

failed_count = len(failed_checks)
passed_count = total_checks - failed_count

print(f"Total checks: {total_checks}")
print(f"Passed checks: {passed_count}")
print(f"Failed checks: {failed_count}")

actual_value,check_name,expected_value,status
26639,bronze_silver_claim_count_reconciliation,26639,PASS
678013,bronze_silver_policy_count_reconciliation,678013,PASS
0,claims_invalid_claim_amounts,0,PASS
0,claims_null_policy_ids,0,PASS
0,claims_without_valid_policy,0,PASS
True,cost_claim_amount,True,PASS
True,cost_exposure,True,PASS
678013,cost_policy_count,678013,PASS
26444,cost_recorded_claims,26444,PASS
36102,cost_reported_claims,36102,PASS


Total checks: 33
Passed checks: 33
Failed checks: 0


In [0]:
# Fail the pipeline if any critical data quality check failed
if failed_count > 0:
    failed_check_names = [
        result["check_name"]
        for result in failed_checks
    ]

    raise RuntimeError(
        f"Data quality validation failed: "
        f"{failed_count} of {total_checks} checks failed. "
        f"Failed checks: {', '.join(failed_check_names)}"
    )

print(
    f"DATA QUALITY PASSED: "
    f"All {total_checks} checks completed successfully."
)

DATA QUALITY PASSED: All 33 checks completed successfully.
